In [30]:
from pyomo.environ import *

In [31]:
input_matrix = [
    [1, 0, 1, 0, 1],
    [0, 0, 0, 0, 0],
    [1, 0, 0, 1, 0],
    [1, 1, 0, 0, 1],
    [1, 1, 1, 1, 1],
]


In [32]:
model = ConcreteModel()

In [33]:
row_len = len(input_matrix)
col_len = len(input_matrix[0])
model.I = RangeSet(row_len)
model.J = RangeSet(col_len)

In [34]:
def translated_matrix(model, i, j):
    return input_matrix[i-1][j-1]

In [35]:
model.x = Var(model.I, model.J, domain=Binary)
model.y = Var(model.I, model.J, domain=NonNegativeIntegers, bounds=(0,5))

In [36]:
model.matrix = Param(model.I, model.J, initialize=translated_matrix)

In [37]:
def cons(model, i, j):
    flips = model.x[i, j]
    
    if i > 1:
        flips += model.x[i-1, j]
    if i < row_len:
        flips += model.x[i+1, j]
    if j > 1:
        flips += model.x[i, j-1]
    if j < col_len: 
        flips += model.x[i, j+1]
    

    return (model.matrix[i,j] + flips) == 2*model.y[i,j]

In [38]:
model.obj = Objective(rule=sum(model.x[i,j] for i in model.I for j in model.J), sense=minimize)

In [39]:
model.cons = Constraint(model.I, model.J, rule=cons)

In [40]:
solver = SolverFactory('glpk')
result = solver.solve(model, tee=False)

In [41]:
x_matrix = [[int(model.x[i,j].value) for j in model.J] for i in model.I]

print("Flip decision matrix:")
for row in x_matrix:
    print(row)

objective_value = model.obj()
print("\n")
print(f"Minimum number of flip actions required: {int(objective_value)}")

Flip decision matrix:
[1, 1, 0, 0, 0]
[1, 0, 0, 0, 1]
[0, 0, 0, 1, 1]
[0, 0, 1, 1, 1]
[1, 0, 0, 0, 0]


Minimum number of flip actions required: 10
